In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

In [2]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [3]:
drop_cols = ["Id","MiscFeature", "PoolQC","FireplaceQu","MasVnrType","Alley","Fence"]
train_df.drop(drop_cols, axis =1, inplace = True)
test_df.drop(drop_cols, axis = 1, inplace = True)

In [4]:
y1 = train_df["SalePrice"]

In [5]:
X = train_df.drop("SalePrice", axis = 1)
X_test = test_df

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(X,y1, test_size =0.2)

In [8]:
cat_cols = X_train.select_dtypes(include = "object").columns
num_cols = X_train.select_dtypes(include = ["float","int"]).columns

In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe, cat_cols),  # merge all cat cols
    ],
    remainder="drop",
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=300,
        objective='reg:squarederror',
        learning_rate = 0.05, 
        random_state=42
    )),
])

pipeline.fit(X_train, y_train)
preds = pipeline.predict(X_valid)
rmse = root_mean_squared_error(preds,y_valid)

In [17]:
rmse

22666.251953125